# 例 7.7 · 按去年同月收益构建标普 500 多空组合

去年同月表现最好的股票，今年同月是否仍会跑赢？本例在每个月末，用去年对应持有月份的个股收益排名，做多最高十分位、做空最低十分位，持有一个月。例如，五月末为六月选股时，使用去年六月的单月收益。[MATLAB 原例](../matlab/example7_7.m) · [Python 实现](../python/examples/example7_7.py)

In [ ]:
import sys
from pathlib import Path

import numpy as np
from scipy.io import loadmat

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.is_last_trading_day_of_month import is_last_trading_day_of_month
from epchan.lag1 import lag1
from epchan.paths import data_path
from epchan.smartmean import smartmean
from epchan.smartstd import smartstd
from epchan.smartsum import smartsum

## 1. 用相邻月末收盘价计算个股月收益

交易日序列中，如果下一交易日属于下个月，当前交易日就是本月的最后一个交易日。取这些日期的收盘价计算简单收益率，保留缺失值。

In [ ]:
data = loadmat(data_path('SPX_20071123.mat'), variable_names=['tday', 'cl'])
dates = data['tday'].ravel().astype(int)
month_ends = np.flatnonzero(is_last_trading_day_of_month(dates))
month_close = data['cl'][month_ends]
previous_close = lag1(month_close)
monthly_returns = (month_close - previous_close) / previous_close
print(f'样本含 {len(month_ends)} 个月末，覆盖 {month_close.shape[1]} 只股票。')

## 2. 按去年同月的收益排名建立持仓

对持有月 `m`，读取 `m - 12` 月的收益排名，将持仓记在 `m - 1` 月末。为复现原例，缺失值筛选沿用源代码的列顺序：它逐列组合**已排序**的历史收益和**原股票顺序**的建仓月末价格的有效性，再用筛选结果取排名中的股票编号。每侧股数为筛选后数量的十分之一，向下取整。

In [ ]:
positions = np.zeros_like(monthly_returns)
for month in range(12, len(month_ends)):
    signal = monthly_returns[month - 12]
    ranked = np.argsort(signal, kind='stable')
    valid = np.isfinite(signal[ranked]) & np.isfinite(month_close[month - 1])
    selected = ranked[valid]
    count = len(selected) // 10
    if count == 0:
        continue
    positions[month - 1, selected[:count]] = -1
    positions[month - 1, selected[-count:]] = 1

## 3. 计算月度损益和年化指标

将上月末持仓乘以本月收益，再按股票求和。原例每只入选股票持有 +1 或 −1 单位名义金额，因此月度指标是各头寸的合计损益。按全样本有效月份计算，年化平均损益等于月均损益乘以 12，年化夏普比率等于月均损益除以月损益的总体标准差，再乘以 $\sqrt{12}$。

In [ ]:
monthly_pnl = smartsum(lag1(positions) * monthly_returns, axis=1)
mean_pnl = smartmean(monthly_pnl, axis=0)
annualized_mean = 12 * mean_pnl
sharpe = np.sqrt(12) * mean_pnl / smartstd(monthly_pnl, axis=0)
print(f'年化平均损益 = {annualized_mean:.4f}')
print(f'年化夏普比率 = {sharpe:.4f}')

### 结果解读

按原例的持仓规模，样本中的年化平均损益为 **−0.9167**，年化夏普比率为 **−0.1055**。多头与空头合计后，去年同月排名信号在这段样本中的平均损益为负。